# Инференс модели характеристик вина

Короткий ноутбук для применения модели, сохранённой основным ноутбуком `01_wine_characteristics_project.ipynb`. Выполните основной ноутбук хотя бы один раз, чтобы создать `artifacts/wine_characteristics_model.joblib`.

In [1]:
from pathlib import Path
import json
import re

import joblib
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "artifacts" / "wine_characteristics_model.joblib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL_PATH = PROJECT_ROOT / "artifacts" / "wine_characteristics_model.joblib"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Сначала выполните notebooks/01_wine_characteristics_project.ipynb")

bundle = joblib.load(MODEL_PATH)
print("Модель загружена:", MODEL_PATH.resolve())
print("Метаданные:", bundle["metadata"])
if bundle["metadata"].get("demo_weak_labels"):
    print("⚠ Это демонстрационная модель на слабой разметке; не используйте её оценки как production-вероятности.")

/Users/sergej/venvs/ds-ml-py311/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_array, issparse


Модель загружена: /Users/sergej/Desktop/УрФУ/Семестр 3/Интеграции ИИ в разработку/project/artifacts/wine_characteristics_model.joblib
Метаданные: {'demo_weak_labels': True, 'n_samples': 42, 'random_state': 42, 'text_columns': ['name', 'description', 'consumption_recommendation']}
⚠ Это демонстрационная модель на слабой разметке; не используйте её оценки как production-вероятности.


In [2]:
def clean_text(value):
    value = "" if value is None else str(value)
    value = value.lower().replace("ё", "е")
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def predict_wine(description, model_bundle=bundle):
    X = model_bundle["vectorizer"].transform([clean_text(description)])
    result = {"categories": {}, "aroma_notes": [], "taste_notes": []}
    for target, clf in model_bundle["categorical_models"].items():
        label = str(clf.predict(X)[0])
        confidence = float(np.max(clf.predict_proba(X)[0])) if hasattr(clf, "predict_proba") else None
        result["categories"][target] = {"label": label, "confidence": confidence}
    for target, target_bundle in model_bundle["multilabel_models"].items():
        notes = []
        for label, clf in target_bundle["models"].items():
            classes = list(clf.classes_)
            probability = float(clf.predict_proba(X)[0, classes.index(1)]) if 1 in classes else 0.0
            if probability >= target_bundle["threshold"]:
                notes.append({"label": str(label), "probability": round(probability, 3)})
        result[target] = sorted(notes, key=lambda x: x["probability"], reverse=True)
    return result

In [3]:
# Замените текст на описание нового вина.
description = '''
Насыщенное красное вино. В аромате черная смородина, вишня, пряности и оттенки дуба.
Вкус богатый, с заметными танинами, ягодными нотами и долгим послевкусием.
'''

prediction = predict_wine(description)
print(json.dumps(prediction, ensure_ascii=False, indent=2))

{
  "categories": {
    "acidity_level": {
      "label": "средняя",
      "confidence": 0.5717014606456161
    },
    "tannin_level": {
      "label": "средняя",
      "confidence": 0.48432158080671567
    },
    "body_level": {
      "label": "полное",
      "confidence": 0.4642449087098258
    },
    "intensity_level": {
      "label": "высокая",
      "confidence": 0.48117747158548324
    }
  },
  "aroma_notes": [
    {
      "label": "черные ягоды",
      "probability": 0.499
    },
    {
      "label": "красные ягоды",
      "probability": 0.48
    },
    {
      "label": "пряности",
      "probability": 0.456
    },
    {
      "label": "травы",
      "probability": 0.42
    },
    {
      "label": "цитрусы",
      "probability": 0.367
    }
  ],
  "taste_notes": [
    {
      "label": "цитрусы",
      "probability": 0.362
    }
  ]
}
